In [61]:
import data_generation_utils as dgu
from greedy_algorithms import modified_greedy

instances = dgu.read_instances("instances.txt")

In [62]:
initial_solution = modified_greedy(instances[0])
initial_solution

(1079,
 [Task(id=4, r=2, l=2, w=7),
  Task(id=7, r=3, l=1, w=5),
  Task(id=9, r=3, l=7, w=10),
  Task(id=11, r=3, l=6, w=9),
  Task(id=0, r=5, l=3, w=7),
  Task(id=1, r=0, l=12, w=10),
  Task(id=6, r=6, l=11, w=8),
  Task(id=10, r=2, l=11, w=7),
  Task(id=2, r=4, l=7, w=5),
  Task(id=8, r=5, l=9, w=5),
  Task(id=3, r=4, l=9, w=4),
  Task(id=5, r=3, l=9, w=1)],
 [(4, 0, 2, 4),
  (7, 1, 3, 4),
  (9, 2, 3, 10),
  (11, 3, 3, 9),
  (0, 0, 5, 8),
  (1, 1, 4, 16),
  (6, 0, 8, 19),
  (10, 3, 9, 20),
  (2, 2, 10, 17),
  (8, 1, 16, 25),
  (3, 2, 17, 26),
  (5, 0, 19, 28)])

In [63]:
def format_solution(solution):
    formatted_solution = [[] for _ in range(instances[0].n_processors)]
    _, _, schedules = initial_solution
    for i in range(len(schedules)):
        task_id, processor_id, _, _ = schedules[i]
        formatted_solution[processor_id].append(instances[0].tasks[task_id])
    
    return formatted_solution
    

In [64]:
initial_solution = format_solution(initial_solution)
initial_solution

[[Task(id=4, r=2, l=2, w=7),
  Task(id=0, r=5, l=3, w=7),
  Task(id=6, r=6, l=11, w=8),
  Task(id=5, r=3, l=9, w=1)],
 [Task(id=7, r=3, l=1, w=5),
  Task(id=1, r=0, l=12, w=10),
  Task(id=8, r=5, l=9, w=5)],
 [Task(id=9, r=3, l=7, w=10),
  Task(id=2, r=4, l=7, w=5),
  Task(id=3, r=4, l=9, w=4)],
 [Task(id=11, r=3, l=6, w=9), Task(id=10, r=2, l=11, w=7)]]

In [65]:
def objective(schedule):
    total = 0

    for processor in schedule:
        current_time = 0

        for task in processor:
            start = max(current_time, task.r)
            completion = start + task.l

            total += task.w * completion
            current_time = completion

    return total

objective(initial_solution)

1079

In [66]:
def local_search(instance, initial_solution):
    current = [processor_load.copy() for processor_load in initial_solution]
    current_value = objective(current)

    improved = True

    # Nearest neighborhood: 
    # looping over swaps of 1 pair of tasks and relocations of 1 task
    
    while improved:
        improved = False

        # try all swaps
        for p1 in range(len(current)):
            for p2 in range(p1, len(current)):

                for i in range(len(current[p1])):
                    for j in range(len(current[p2])):

                        if p1 == p2 and i == j:
                            continue

                        # swap
                        current[p1][i], current[p2][j] = (
                            current[p2][j],
                            current[p1][i]
                        )

                        new_value = objective(current)

                        if new_value < current_value:
                            current_value = new_value
                            improved = True
                            break

                        # undo
                        current[p1][i], current[p2][j] = (
                            current[p2][j],
                            current[p1][i]
                        )

                    if improved:
                        break

                if improved:
                    break

            if improved:
                break

        if improved:
            continue

#         try all relocations:
        for p1 in range(len(current)):
            for i in range(len(current[p1])):

                task = current[p1][i]

                for p2 in range(len(current)):
                    for j in range(len(current[p2]) + 1):

                        if p1 == p2:
                            continue

                        # Remove task
                        current[p1].pop(i)

                        # Insert into another processor
                        current[p2].insert(j, task)

                        new_value = objective(current)

                        if new_value < current_value:
                            current_value = new_value
                            improved = True
                            break

                        # Undo
                        current[p2].pop(j)
                        current[p1].insert(i, task)

                    if improved:
                        break

                if improved:
                    break

            if improved:
                break

    return current


objective(local_search(instances[0], initial_solution))

1022

In [67]:
import random

def random_swap(sol):
    """swap two randomly chosen tasks (possibly on the same processor)."""
    non_empty = [i for i in range(len(sol)) if sol[i]]
    if not non_empty:
        return
    p1 = random.choice(non_empty)
    p2 = random.choice(non_empty)
    i = random.randrange(len(sol[p1]))
    j = random.randrange(len(sol[p2]))
    if p1 == p2 and i == j:
        return
    sol[p1][i], sol[p2][j] = sol[p2][j], sol[p1][i]
    
    
print(initial_solution)
random_swap(initial_solution)
print(initial_solution)

[[Task(id=4, r=2, l=2, w=7), Task(id=0, r=5, l=3, w=7), Task(id=6, r=6, l=11, w=8), Task(id=5, r=3, l=9, w=1)], [Task(id=7, r=3, l=1, w=5), Task(id=1, r=0, l=12, w=10), Task(id=8, r=5, l=9, w=5)], [Task(id=9, r=3, l=7, w=10), Task(id=2, r=4, l=7, w=5), Task(id=3, r=4, l=9, w=4)], [Task(id=11, r=3, l=6, w=9), Task(id=10, r=2, l=11, w=7)]]
[[Task(id=4, r=2, l=2, w=7), Task(id=0, r=5, l=3, w=7), Task(id=6, r=6, l=11, w=8), Task(id=5, r=3, l=9, w=1)], [Task(id=7, r=3, l=1, w=5), Task(id=1, r=0, l=12, w=10), Task(id=8, r=5, l=9, w=5)], [Task(id=10, r=2, l=11, w=7), Task(id=2, r=4, l=7, w=5), Task(id=3, r=4, l=9, w=4)], [Task(id=11, r=3, l=6, w=9), Task(id=9, r=3, l=7, w=10)]]


In [68]:
def random_relocate(sol):
    """move one randomly chosen task to a random position on a random processor."""
    non_empty = [i for i in range(len(sol)) if sol[i]]
    if not non_empty:
        return
    p1 = random.choice(non_empty)
    i = random.randrange(len(sol[p1]))
    task = sol[p1].pop(i)
    p2 = random.randrange(len(sol))
    j = random.randrange(len(sol[p2]) + 1)
    sol[p2].insert(j, task)

print(initial_solution)
random_relocate(initial_solution)
print(initial_solution)


[[Task(id=4, r=2, l=2, w=7), Task(id=0, r=5, l=3, w=7), Task(id=6, r=6, l=11, w=8), Task(id=5, r=3, l=9, w=1)], [Task(id=7, r=3, l=1, w=5), Task(id=1, r=0, l=12, w=10), Task(id=8, r=5, l=9, w=5)], [Task(id=10, r=2, l=11, w=7), Task(id=2, r=4, l=7, w=5), Task(id=3, r=4, l=9, w=4)], [Task(id=11, r=3, l=6, w=9), Task(id=9, r=3, l=7, w=10)]]
[[Task(id=4, r=2, l=2, w=7), Task(id=0, r=5, l=3, w=7), Task(id=6, r=6, l=11, w=8), Task(id=5, r=3, l=9, w=1)], [Task(id=7, r=3, l=1, w=5), Task(id=1, r=0, l=12, w=10), Task(id=8, r=5, l=9, w=5), Task(id=2, r=4, l=7, w=5)], [Task(id=10, r=2, l=11, w=7), Task(id=3, r=4, l=9, w=4)], [Task(id=11, r=3, l=6, w=9), Task(id=9, r=3, l=7, w=10)]]


In [69]:
def random_shuffle_all(sol):
    """Strongest perturbation: flatten all tasks and redistribute randomly."""
    all_tasks = [task for proc in sol for task in proc]
    random.shuffle(all_tasks)
    m = len(sol)
    new_sol = [[] for _ in range(m)]
    for idx, task in enumerate(all_tasks):
        new_sol[idx % m].append(task)
    return new_sol


In [70]:
def shake(solution, k):
    """
    shake the solution
    strength increases with k:
      k=1: 1 random swap
      k=2: 1 random relocation
      k=3: 2 random swaps
      k=4: 2 random relocations
      k=5: 3 relocations + 2 swaps
    """
    new_solution = [p.copy() for p in solution]

    if k == 1:
        random_swap(new_solution)
    elif k == 2:
        random_relocate(new_solution)
    elif k == 3:
        random_swap(new_solution)
        random_swap(new_solution)
    elif k == 4:
        random_relocate(new_solution)
        random_relocate(new_solution)
    elif k == 5:
        for _ in range(3):
            random_relocate(new_solution)
        for _ in range(2):
            random_swap(new_solution)
    else:
        new_solution = random_shuffle_all(new_solution)
    
    return new_solution

print(initial_solution)
shake(initial_solution, 5)
initial_solution

[[Task(id=4, r=2, l=2, w=7), Task(id=0, r=5, l=3, w=7), Task(id=6, r=6, l=11, w=8), Task(id=5, r=3, l=9, w=1)], [Task(id=7, r=3, l=1, w=5), Task(id=1, r=0, l=12, w=10), Task(id=8, r=5, l=9, w=5), Task(id=2, r=4, l=7, w=5)], [Task(id=10, r=2, l=11, w=7), Task(id=3, r=4, l=9, w=4)], [Task(id=11, r=3, l=6, w=9), Task(id=9, r=3, l=7, w=10)]]


[[Task(id=4, r=2, l=2, w=7),
  Task(id=0, r=5, l=3, w=7),
  Task(id=6, r=6, l=11, w=8),
  Task(id=5, r=3, l=9, w=1)],
 [Task(id=7, r=3, l=1, w=5),
  Task(id=1, r=0, l=12, w=10),
  Task(id=8, r=5, l=9, w=5),
  Task(id=2, r=4, l=7, w=5)],
 [Task(id=10, r=2, l=11, w=7), Task(id=3, r=4, l=9, w=4)],
 [Task(id=11, r=3, l=6, w=9), Task(id=9, r=3, l=7, w=10)]]

In [71]:
def vns(instance, initial_solution, k_max=6, max_iterations=200,
        max_no_improve=None, verbose=False):
    """
    VNS: shake with increasing neighborhood strength k, run local
    search on the shaken solution, and re-center on it (and reset k) whenever
    it beats the current best; otherwise move to the next neighborhood.
 
    Parameters
    ----------
    instance : whatever local_search expects as instance data (unused by the
        objective/local_search shown, but kept for interface compatibility)
    initial_solution : list[list[Task]]
    k_max : largest neighborhood index to try before giving up on a round
    max_iterations : cap on outer VNS rounds
    max_no_improve : stop early after this many rounds with no improvement
    verbose : print progress
 
    Returns
    -------
    list[list[Task]] : best schedule found
    """
    best = [p.copy() for p in initial_solution]
    best_value = objective(best)
    no_improve_count = 0
    iteration = 0
    
    while iteration < max_iterations:
        if max_no_improve is not None and no_improve_count >= max_no_improve:
            break
        improved_this_round = False
        k = 1
        while k <= k_max:
            shaken = shake(best, k)
            candidate = local_search(instance, shaken)
            candidate_value = objective(candidate)
            if candidate_value < best_value:
                best = candidate
                best_value = candidate_value
                improved_this_round = True
                if verbose:
                    print(f"iter {iteration}: improved -> {best_value} (k={k})")
                k = 1  # re-center and restart from the smallest neighborhood
            else:
                k += 1
        no_improve_count = 0 if improved_this_round else no_improve_count + 1
        iteration += 1
    if verbose:
        print(f"VNS done. Best objective: {best_value}")
    return best


In [72]:
instances = dgu.read_instances("instances.txt")

initial_solution = modified_greedy(instances[0])
initial_solution = format_solution(initial_solution)
ls_solution = local_search(instances[0], initial_solution)
vns_solution = vns(instances[0], ls_solution, max_iterations=1000, verbose=True)

objective(vns_solution)


VNS done. Best objective: 1022


1022

In [73]:
def random_move(sol):
    """
    apply one random move to "sol" in place and return a zero-arg function
    that undoes it. Move type is chosen 50/50 between swap and relocate.
    """
    non_empty = [i for i in range(len(sol)) if sol[i]]
    if not non_empty:
        return lambda: None

    if random.random() < 0.5:
        # swap
        p1 = random.choice(non_empty)
        p2 = random.choice(non_empty)
        i = random.randrange(len(sol[p1]))
        j = random.randrange(len(sol[p2]))
        if p1 == p2 and i == j:
            return lambda: None
        sol[p1][i], sol[p2][j] = sol[p2][j], sol[p1][i]

        def undo():
            sol[p1][i], sol[p2][j] = sol[p2][j], sol[p1][i]

        return undo

    else:
        # relocate
        p1 = random.choice(non_empty)
        i = random.randrange(len(sol[p1]))
        task = sol[p1].pop(i)
        p2 = random.randrange(len(sol))
        j = random.randrange(len(sol[p2]) + 1)
        sol[p2].insert(j, task)

        def undo():
            sol[p2].pop(j)
            sol[p1].insert(i, task)

        return undo

In [74]:
import math

def simulated_annealing(instance, initial_solution,
                         t0=1000.0, t_min=1e-3, alpha=0.95,
                         iters_per_temp=50, max_iterations=10000, verbose=False):
    """
    Simulated annealing for weighted completion time scheduling.

    t0 / t_min / alpha : geometric cooling from t0 down to t_min, T *= alpha
        every `iters_per_temp` moves
    max_iterations : hard cap on total moves evaluated
    """

    current = [p.copy() for p in initial_solution]
    current_value = objective(current)

    best = [p.copy() for p in current]
    best_value = current_value

    T = t0
    total_moves = 0

    while (T > t_min and total_moves < max_iterations):

        for _ in range(iters_per_temp):
            if max_iterations is not None and total_moves >= max_iterations:
                break

            undo = random_move(current)
            new_value = objective(current)
            delta = new_value - current_value
            total_moves += 1

            if delta < 0 or random.random() < math.exp(-delta / T):
                # accept (improving move, or accepted worsening move)
                current_value = new_value
                if current_value < best_value:
                    best = [p.copy() for p in current]
                    best_value = current_value
                    if verbose:
                        print(f"T={T:.3f} move={total_moves}: new best {best_value}")
            else:
                # reject: undo the move
                undo()

        T *= alpha

    if verbose:
        print(f"SA done. Best objective: {best_value}")

    return best


initial_solution = modified_greedy(instances[0])
initial_solution = format_solution(initial_solution)
sa_solution = simulated_annealing(
    instances[0], initial_solution,
    t0=1000.0, t_min=1e-3, alpha=0.9, iters_per_temp=50, verbose=False,
)
sa_value = objective(sa_solution)
print("SA:", sa_value)
sa_solution


SA: 1022


[[Task(id=11, r=3, l=6, w=9),
  Task(id=6, r=6, l=11, w=8),
  Task(id=5, r=3, l=9, w=1)],
 [Task(id=1, r=0, l=12, w=10), Task(id=8, r=5, l=9, w=5)],
 [Task(id=9, r=3, l=7, w=10), Task(id=10, r=2, l=11, w=7)],
 [Task(id=4, r=2, l=2, w=7),
  Task(id=7, r=3, l=1, w=5),
  Task(id=0, r=5, l=3, w=7),
  Task(id=2, r=4, l=7, w=5),
  Task(id=3, r=4, l=9, w=4)]]

In [75]:
greedy_sol = modified_greedy(instances[0])
greedy_sol

(1079,
 [Task(id=4, r=2, l=2, w=7),
  Task(id=7, r=3, l=1, w=5),
  Task(id=9, r=3, l=7, w=10),
  Task(id=11, r=3, l=6, w=9),
  Task(id=0, r=5, l=3, w=7),
  Task(id=1, r=0, l=12, w=10),
  Task(id=6, r=6, l=11, w=8),
  Task(id=10, r=2, l=11, w=7),
  Task(id=2, r=4, l=7, w=5),
  Task(id=8, r=5, l=9, w=5),
  Task(id=3, r=4, l=9, w=4),
  Task(id=5, r=3, l=9, w=1)],
 [(4, 0, 2, 4),
  (7, 1, 3, 4),
  (9, 2, 3, 10),
  (11, 3, 3, 9),
  (0, 0, 5, 8),
  (1, 1, 4, 16),
  (6, 0, 8, 19),
  (10, 3, 9, 20),
  (2, 2, 10, 17),
  (8, 1, 16, 25),
  (3, 2, 17, 26),
  (5, 0, 19, 28)])